## **Phase 4: Data Modelling**
This step is to convert the cleaned files into a consistent analytical model.

In [1]:
# Importing the necessary libraries
import pandas as pd
import fastf1 as f1
from pathlib import Path
import os

### Dimension Tables

#### 1. Dim Driver

In [45]:
# Creating a dictionary where the data will be fetched from results data as driver ID to be the key and a list of values as their value.
# The dictionary will make sure that the values aren't repeated.
# Once the dictionary is created, the dictionary will be saved in a Pandas dataframe as Dim_Driver

directory_folder = "f1_cleaned_data"
driver_dict = {}

# Step 1: Fetching the values from each tables into a dictionary
for file_path in Path(directory_folder).glob("202*/Round_*/*"):
    results_file = f"{file_path}/results_cleaned.parquet"
    df = pd.read_parquet(results_file)
    for index, row in df.iterrows():
        full_name_split = row['FullName'].split(" ")
        first_name = " ".join(full_name_split[:-1])
        last_name = full_name_split[-1]
        if row['Abbreviation'] not in driver_dict:
            driver_dict[row['Abbreviation']] = [
                row['DriverNumber'], row['DriverId'], first_name, last_name, row['FullName'], row['CountryCode']
            ]

# Step 2: For 2025 season, the CountryCode is not there in the table. To fix that, a manual dictionary is added to fetch the CountryCode of the drivers who got in F1 in 2025.
country_code_dict_2025 = {'NOR': 'GBR', 'VER': 'NED', 'RUS': 'GBR', 'ANT': 'ITA', 'ALB': 'THA', 'STR': 'CAN', 'HUL': 'GER', 'LEC': 'MON', 'PIA': 'AUS', 'HAM': 'GBR', 'GAS': 'FRA', 'TSU': 'JPN', 'OCO': 'FRA', 'BEA': 'GBR','LAW': 'NZL', 'BOR': 'BRA', 'ALO': 'ESP', 'SAI': 'ESP', 'DOO': 'AUS', 'HAD': 'FRA', 'COL': 'ARG'}

# Step 3: Adding the 2025 drivers' CountryCode into the original dictionary
for k1 in driver_dict.keys():
    for k2 in country_code_dict_2025.keys():
        if k1==k2:
            driver_dict[k1][-1] = country_code_dict_2025[k2]

# Step 4: Converting the dictionary into the Pandas Dataframe
columns = ["driver_id", "driver_number", "driver_code", "first_name", "last_name", "full_name", "country_code"]
dim_driver = pd.DataFrame(
    [[key, *values] for key, values in driver_dict.items()], columns=columns
)
dim_driver

,driver_id,driver_number,driver_code,first_name,last_name,full_name,country_code
0,SAI,55,sainz,Carlos,Sainz,Carlos Sainz,ESP
1,PER,11,perez,Sergio,Perez,Sergio Perez,MEX
2,HAM,44,hamilton,Lewis,Hamilton,Lewis Hamilton,GBR
3,LEC,16,leclerc,Charles,Leclerc,Charles Leclerc,MON
4,ALO,14,alonso,Fernando,Alonso,Fernando Alonso,ESP
5,NOR,4,norris,Lando,Norris,Lando Norris,GBR
6,VER,1,verstappen,Max,Verstappen,Max Verstappen,NED
7,MSC,47,schumacher,Mick,Schumacher,Mick Schumacher,GER
8,VET,5,vettel,Sebastian,Vettel,Sebastian Vettel,GER
9,MAG,20,magnussen,Kevin,Magnussen,Kevin Magnussen,DEN


#### 2. Dim Constructor

In [61]:
# We will use the same technique which was used to create the dim_driver.
# TeamID will be used as constructor_id and a hardcoded dictionary will be used to provide the constructor's nationality

directory_folder = "f1_cleaned_data"
constructor_dict = {}

# Step 1: Fetching the values from each tables into a dictionary
for file_path in Path(directory_folder).glob("202*/Round_*/*"):
    results_file = f"{file_path}/results_cleaned.parquet"
    df = pd.read_parquet(results_file)
    for index, row in df.iterrows():
        if row['TeamId'] not in constructor_dict:
            constructor_dict[row['TeamId']] = [row['TeamName'], row['TeamColor'], "constructor_nationality"]

# Step 2: A manual dictionary to add nationality of each constructors.
constructor_nationality = {'alpine': 'France', 'aston_martin': 'United Kingdom', 'ferrari': 'Italy', 'haas': 'United States', 'mclaren': 'United Kingdom', 'mercedes': 'Germany', 'rb': 'Italy', 'red_bull': 'Austria', 'williams': 'United Kingdom', 'sauber': 'Switzerland', 'alfa': 'Switzerland', 'alphatauri': 'Italy'}

# Step 3: Adding the Nationality to the constructors
for k1 in constructor_dict.keys():
    for k2 in constructor_nationality.keys():
        if k1 == k2:
            constructor_dict[k1][-1] = constructor_nationality[k2]

# Step 4: Converting the dictionary into the Pandas Dataframe
columns = ['constructor_id', 'constructor_name', 'constructor_color', 'nationality']
dim_constructor = pd.DataFrame(
    [[key, *values] for key, values in constructor_dict.items()], columns=columns
)

# Step 5: create a standardized name column in dim_constructor
dim_constructor['standardised_name'] = dim_constructor['constructor_name'].str.replace(" ", "_")
con_stan = {'Haas_F1_Team': 'Haas', 'RB': 'Racing_Bulls'}
dim_constructor['standardised_name'] = dim_constructor['standardised_name'].map(con_stan).fillna(dim_constructor['standardised_name'])

dim_constructor


,constructor_id,constructor_name,constructor_color,nationality,standardised_name
0,ferrari,Ferrari,ed1c24,Italy,Ferrari
1,red_bull,Red Bull Racing,1e5bc6,Austria,Red_Bull_Racing
2,mercedes,Mercedes,6cd3bf,Germany,Mercedes
3,alpine,Alpine,2293d1,France,Alpine
4,mclaren,McLaren,f58020,United Kingdom,McLaren
5,haas,Haas F1 Team,b6babd,United States,Haas
6,aston_martin,Aston Martin,2d826d,United Kingdom,Aston_Martin
7,williams,Williams,37bedd,United Kingdom,Williams
8,alphatauri,AlphaTauri,4e7c9b,Italy,AlphaTauri
9,alfa,Alfa Romeo,b12039,Switzerland,Alfa_Romeo
